# Extract

In [1]:
import pandas as pd
import time

In [2]:
path_main = 'https://github.com/KhalPrawira/Big-Data_Assignment/raw/refs/heads/main/UAS/Dataset/Pakistans%20Largest%20E-Commerce%20Dataset.csv'
path_holiday = 'https://github.com/KhalPrawira/Big-Data_Assignment/raw/refs/heads/main/UAS/Dataset/pakistan_holiday.csv'

def step_1_extract_raw():
    print("=== MULAI TAHAP 1: EXTRACT ===")
    
    # 1. BACA DATA UTAMA (E-COMMERCE)
    print("[1/3] Membaca file CSV Utama...")
    try:
        df_ecommerce = pd.read_csv(path_main, low_memory=False)
        print(f"   -> Sukses. Ukuran: {df_ecommerce.shape}")
        print("   -> Catatan: Kolom sampah (Unnamed) dan tanggal mentah ikut terbaca.")
    except Exception as e:
        print(f"   -> ERROR: {e}")
        return None, None, None

    # 2. BACA DATA KEDUA (HOLIDAY)
    print("[2/3] Membaca file CSV Holiday...")
    try:
        df_holiday = pd.read_csv(path_holiday)
        print(f"   -> Sukses. Ukuran: {df_holiday.shape}")
    except Exception as e:
        print(f"   -> ERROR: {e}")
        return None, None, None

    # 3. BUAT DATA KETIGA (JSON MAPPING)
    print("[3/3] Membuat Data Dummy JSON (Kategori)...")
    data_kategori = [
        {"category": "Mobiles & Tablets", "group": "Electronics"},
        {"category": "Superstore", "group": "Groceries"},
        {"category": "Women's Fashion", "group": "Fashion"},
        {"category": "Men's Fashion", "group": "Fashion"},
        {"category": "Appliances", "group": "Home & Living"},
        {"category": "Computing", "group": "Electronics"},
        {"category": "Beauty & Grooming", "group": "Health & Beauty"},
        {"category": "Soghaat", "group": "Gifts/Soghaat"},
        {"category": "Kids & Baby", "group": "Kids"},
        {"category": "Others", "group": "Others"}
    ]
    df_category_map = pd.DataFrame(data_kategori)
    print(f"   -> Sukses. Ukuran: {df_category_map.shape}")

    print("\n=== EXTRACT SELESAI ===")
    print("Data tersimpan di memori (variabel). JANGAN RESTART KERNEL sebelum lanjut ke tahap Load.")
    
    return df_ecommerce, df_holiday, df_category_map

df_raw_ecommerce, df_raw_holiday, df_raw_category = step_1_extract_raw()

=== MULAI TAHAP 1: EXTRACT ===
[1/3] Membaca file CSV Utama...
   -> Sukses. Ukuran: (1153432, 26)
   -> Catatan: Kolom sampah (Unnamed) dan tanggal mentah ikut terbaca.
[2/3] Membaca file CSV Holiday...
   -> Sukses. Ukuran: (60, 5)
[3/3] Membuat Data Dummy JSON (Kategori)...
   -> Sukses. Ukuran: (10, 2)

=== EXTRACT SELESAI ===
Data tersimpan di memori (variabel). JANGAN RESTART KERNEL sebelum lanjut ke tahap Load.


In [3]:
df_raw_ecommerce.head()

,item_id,status,created_at,sku,price,qty_ordered,grand_total,increment_id,category_name_1,sales_commission_code,...,Month,Customer Since,M-Y,FY,Customer ID,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,211131.0,complete,7/1/2016,kreations_YI 06-L,1950.0,1.0,1950.0,100147443,Women's Fashion,\N,...,7.0,2016-7,7-2016,FY17,1.0,NaN,NaN,NaN,NaN,NaN
1,211133.0,canceled,7/1/2016,kcc_Buy 2 Frey Air Freshener & Get 1 Kasual Bo...,240.0,1.0,240.0,100147444,Beauty & Grooming,\N,...,7.0,2016-7,7-2016,FY17,2.0,NaN,NaN,NaN,NaN,NaN
2,211134.0,canceled,7/1/2016,Ego_UP0017-999-MR0,2450.0,1.0,2450.0,100147445,Women's Fashion,\N,...,7.0,2016-7,7-2016,FY17,3.0,NaN,NaN,NaN,NaN,NaN
3,211135.0,complete,7/1/2016,kcc_krone deal,360.0,1.0,60.0,100147446,Beauty & Grooming,R-FSD-52352,...,7.0,2016-7,7-2016,FY17,4.0,NaN,NaN,NaN,NaN,NaN
4,211136.0,order_refunded,7/1/2016,BK7010400AG,555.0,2.0,1110.0,100147447,Soghaat,\N,...,7.0,2016-7,7-2016,FY17,5.0,NaN,NaN,NaN,NaN,NaN


# Load

In [4]:
from sqlalchemy import create_engine, text

## Koneksi ke Database

In [5]:
DB_CONFIG = {
    "user": "admin_k",
    "password": "My name is not Billy nor Alex",
    "host": "localhost",
    "port": "5432",
    "dbname": "ecommerce_dw"
}

connection_str = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
engine = create_engine(connection_str)

try:
    with engine.connect() as conn:
        print("✅ KONEKSI BERHASIL!")
        print(f"Terhubung ke database: {DB_CONFIG['dbname']}")
except Exception as e:
    print("❌ KONEKSI GAGAL!")
    print(f"Error: {e}")
    print("Pastikan container Docker sudah menyala ('Database Running').")

✅ KONEKSI BERHASIL!
Terhubung ke database: ecommerce_dw


## Load Dataset

### 1. Dataset Utama (E-Commerce)

In [6]:
print("--- [1/3] UPLOAD DATA E-COMMERCE ---")
start_time = time.time()

try:
    if 'df_raw_ecommerce' in locals():
        df_raw_ecommerce.to_sql('raw_ecommerce', engine, if_exists='replace', index=False)
        
        rows = len(df_raw_ecommerce)
        cols = len(df_raw_ecommerce.columns)
        print(f"✅ Sukses upload tabel 'raw_ecommerce'.")
        print(f"   Metadata: {rows} baris, {cols} kolom.")
    else:
        print("❌ Error: Variabel data tidak ditemukan. Jalankan Tahap Extract dulu!")
except Exception as e:
    print(f"❌ Gagal upload: {e}")

print(f"Waktu eksekusi: {time.time() - start_time:.2f} detik")

--- [1/3] UPLOAD DATA E-COMMERCE ---
✅ Sukses upload tabel 'raw_ecommerce'.
   Metadata: 1153432 baris, 26 kolom.
Waktu eksekusi: 83.32 detik


### 2. Dataset 2 (Holiday)

In [7]:
print("--- [2/3] UPLOAD DATA HOLIDAY ---")

try:
    if 'df_raw_holiday' in locals():
        df_raw_holiday.to_sql('raw_holiday', engine, if_exists='replace', index=False)
        print(f"✅ Sukses upload tabel 'raw_holiday'.")
        print(f"   Metadata: {len(df_raw_holiday)} baris.")
    else:
        print("❌ Error: Variabel data holiday tidak ditemukan.")
except Exception as e:
    print(f"❌ Gagal upload: {e}")

--- [2/3] UPLOAD DATA HOLIDAY ---
✅ Sukses upload tabel 'raw_holiday'.
   Metadata: 60 baris.


### 3. Dataset Tambahan (JSON)

In [8]:
print("--- [3/3] UPLOAD DATA CATEGORY MAP ---")

try:
    if 'df_raw_category' in locals():
        df_raw_category.to_sql('raw_category_map', engine, if_exists='replace', index=False)
        print(f"✅ Sukses upload tabel 'raw_category_map'.")
        print(f"   Metadata: {len(df_raw_category)} baris.")
    else:
        print("❌ Error: Variabel data kategori tidak ditemukan.")
except Exception as e:
    print(f"❌ Gagal upload: {e}")

--- [3/3] UPLOAD DATA CATEGORY MAP ---
✅ Sukses upload tabel 'raw_category_map'.
   Metadata: 10 baris.


## Validasi isi Database

In [9]:
print("--- VALIDASI FINAL: ISI DATABASE ---")

try:
    with engine.connect() as conn:
        print("\n[METADATA UKURAN DATA]")
        tables = ['raw_ecommerce', 'raw_holiday', 'raw_category_map']
        
        for tbl in tables:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}"))
            count = result.scalar()
            print(f" -> Tabel '{tbl}': {count} baris")

        # Tampilkan Sampel Data Utama (Metadata Struktur)
        print("\n[PREVIEW TABEL UTAMA: raw_ecommerce ]")
        df_preview = pd.read_sql("SELECT * FROM raw_ecommerce LIMIT 5", conn)
        try:
            display(df_preview)
        except:
            print(df_preview)

        print("\n✅ LOAD SELESAI. Data tersimpan di PostgreSQL.")

except Exception as e:
    print(f"❌ Error saat preview: {e}")


--- VALIDASI FINAL: ISI DATABASE ---

[METADATA UKURAN DATA]
 -> Tabel 'raw_ecommerce': 1153432 baris
 -> Tabel 'raw_holiday': 60 baris
 -> Tabel 'raw_category_map': 10 baris

[PREVIEW TABEL UTAMA: raw_ecommerce ]


,item_id,status,created_at,sku,price,qty_ordered,grand_total,increment_id,category_name_1,sales_commission_code,...,Month,Customer Since,M-Y,FY,Customer ID,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,211131.0,complete,7/1/2016,kreations_YI 06-L,1950.0,1.0,1950.0,100147443,Women's Fashion,\N,...,7.0,2016-7,7-2016,FY17,1.0,None,None,None,None,None
1,211133.0,canceled,7/1/2016,kcc_Buy 2 Frey Air Freshener & Get 1 Kasual Bo...,240.0,1.0,240.0,100147444,Beauty & Grooming,\N,...,7.0,2016-7,7-2016,FY17,2.0,None,None,None,None,None
2,211134.0,canceled,7/1/2016,Ego_UP0017-999-MR0,2450.0,1.0,2450.0,100147445,Women's Fashion,\N,...,7.0,2016-7,7-2016,FY17,3.0,None,None,None,None,None
3,211135.0,complete,7/1/2016,kcc_krone deal,360.0,1.0,60.0,100147446,Beauty & Grooming,R-FSD-52352,...,7.0,2016-7,7-2016,FY17,4.0,None,None,None,None,None
4,211136.0,order_refunded,7/1/2016,BK7010400AG,555.0,2.0,1110.0,100147447,Soghaat,\N,...,7.0,2016-7,7-2016,FY17,5.0,None,None,None,None,None



✅ LOAD SELESAI. Data tersimpan di PostgreSQL.


# Transform

## Transform Dimensi waktu

In [10]:
print("--- CEK STRUKTUR TABEL RAW_HOLIDAY ---")
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM raw_holiday LIMIT 0"))
    print("Nama Kolom Asli:", result.keys())

--- CEK STRUKTUR TABEL RAW_HOLIDAY ---
Nama Kolom Asli: RMKeyView(['ADM_name', 'ISO3', 'Date', 'Name', 'Type'])


In [11]:
print("--- INSPEKSI DATA TANGGAL ---")
with engine.connect() as conn:
    print("\n1. Sampel Tanggal di E-COMMERCE (Tabel Utama):")
    res_main = conn.execute(text("SELECT DISTINCT created_at FROM raw_ecommerce LIMIT 5"))
    for row in res_main:
        print(f"   -> {row[0]}")

    print("\n2. Sampel Tanggal di HOLIDAY (Tabel Libur):")
    res_hol = conn.execute(text('SELECT DISTINCT "Date" FROM raw_holiday LIMIT 5'))
    for row in res_hol:
        print(f"   -> {row[0]}")

--- INSPEKSI DATA TANGGAL ---

1. Sampel Tanggal di E-COMMERCE (Tabel Utama):
   -> 10/10/2016
   -> 10/10/2017
   -> 10/11/2016
   -> 10/11/2017
   -> 10/1/2016

2. Sampel Tanggal di HOLIDAY (Tabel Libur):
   -> 2017-11-09
   -> 2016-07-09
   -> 2017-03-13
   -> 2016-10-12
   -> 2016-09-12


In [12]:
print("--- TRANSFORM PART 1: DIMENSI WAKTU ---")
start_time = time.time()

with engine.connect() as conn:
    # 1. SETUP SCHEMA
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS dwh;"))

    # 2. CREATE DIM_DATE
    query_dim_date = """
    DROP TABLE IF EXISTS dwh.dim_date CASCADE;
    
    CREATE TABLE dwh.dim_date AS
    SELECT DISTINCT
        -- 1. Cleaning E-Commerce (Coba Format YYYY-MM-DD standar Python)
        CAST(created_at AS DATE) as date_id,
        
        -- 2. Extraction
        EXTRACT(YEAR FROM CAST(created_at AS DATE)) as year,
        EXTRACT(MONTH FROM CAST(created_at AS DATE)) as month,
        EXTRACT(DAY FROM CAST(created_at AS DATE)) as day,
        TO_CHAR(CAST(created_at AS DATE), 'Day') as day_name,
        
        -- 3. Enrichment
        CASE 
            WHEN h."Name" IS NOT NULL THEN TRUE 
            ELSE FALSE 
        END as is_holiday,
        h."Name" as holiday_name
        
    FROM raw_ecommerce e
    -- JOIN: Kita ubah juga format Holiday menjadi DATE biasa
    -- (Postgres biasanya pintar mendeteksi format angka yyyy-mm-dd secara otomatis)
    LEFT JOIN raw_holiday h ON CAST(e.created_at AS DATE) = CAST(h."Date" AS DATE)
    WHERE created_at IS NOT NULL;
    """
    
    try:
        conn.execute(text(query_dim_date))
        conn.commit()
        print("✅ [SUKSES] Tabel 'dwh.dim_date' BERHASIL dibuat.")
        
        # Validasi
        res = conn.execute(text("SELECT count(*) FROM dwh.dim_date"))
        print(f"   -> Total Tanggal Unik: {res.scalar()}")
        
    except Exception as e:
        print(f"❌ ERROR: {e}")
        print("Jika error 'Date format not recognized', berarti formatnya memang unik.")

print(f"Waktu eksekusi: {time.time() - start_time:.2f} detik")

--- TRANSFORM PART 1: DIMENSI WAKTU ---
✅ [SUKSES] Tabel 'dwh.dim_date' BERHASIL dibuat.
   -> Total Tanggal Unik: 791
Waktu eksekusi: 1.50 detik
